# 🔬 Auroraa Forensics — Training Pipeline

This notebook covers the full ML training loop for the image forensics system:

1. **Feature Extraction** — Run ELA, EXIF, Noise, FFT on every sample image
2. **Data Exploration** — Visualise each signal's distribution for authentic vs manipulated
3. **Model Training** — Train a Random Forest classifier on the extracted features
4. **Evaluation** — Accuracy, Precision, Recall, Confusion Matrix, ROC curve
5. **Export** — Save model to `models/forensics_model.joblib`

**Directory layout expected:**
```
auroraa vault/
├── samples/
│   ├── original/    ← AUTHENTIC images (.jpg/.png/.webp)
│   └── tampered/    ← MANIPULATED images (.jpg/.png/.webp)
├── notebooks/
│   └── training_pipeline.ipynb   ← this file
└── models/
    └── forensics_model.joblib    ← output of Cell 6
```

In [ ]:
# ── 0. Imports & path setup ─────────────────────────────────────────────────
import sys, json, warnings
from pathlib import Path

# Make project root importable
ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
import joblib

from app.image_forensics.core.ela   import compute_ela
from app.image_forensics.core.exif  import extract_exif
from app.image_forensics.core.noise import noise_consistency_map
from app.image_forensics.core.fft   import fft_spectrum_analysis

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")

ORIGINAL_DIR = ROOT / "samples" / "original"
TAMPERED_DIR = ROOT / "samples" / "tampered"
MODEL_DIR    = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}
print(f"Project root : {ROOT}")
print(f"Original     : {len(list(ORIGINAL_DIR.glob('*')))} files")
print(f"Tampered     : {len(list(TAMPERED_DIR.glob('*')))} files")

## 1. Feature Extraction

We extract a **fixed-length feature vector** from each image using the four forensic modules.

| Feature | Source | Description |
|---|---|---|
| `ela_mean` | ELA | Average ELA error across all pixels |
| `ela_max` | ELA | Maximum ELA error |
| `ela_std` | ELA | Std of ELA — high = spatially inconsistent |
| `ela_regional_variance` | ELA | Coefficient of variation across 3×3 grid tiles |
| `ela_suspicious_pct` | ELA | % of pixels with error > mean + 2σ |
| `exif_risk_score` | EXIF | Encoded risk level: low=0.1, medium=0.55, high=0.9, unknown=0.5 |
| `exif_flag_count` | EXIF | Number of forensic flags raised |
| `exif_has_make_model` | EXIF | 1 if Make & Model present, else 0 |
| `exif_timestamp_mismatch` | EXIF | 1 if DateTime ≠ DateTimeOriginal |
| `noise_median_sigma` | Noise | Median noise level across all tiles |
| `noise_std_sigma` | Noise | Std of tile noise — high = inconsistent sensor noise |
| `noise_suspicious_pct` | Noise | % of tiles with noise > median + 2σ |
| `fft_high_freq_ratio` | FFT | Energy in top 25% spatial frequencies / total |
| `fft_peak_freq_bin` | FFT | Bin index of peak power in radial spectrum |


In [ ]:
# ── 1. Feature Extraction ────────────────────────────────────────────────────

EXIF_RISK_MAP = {"low": 0.1, "medium": 0.55, "high": 0.9, "unknown": 0.5}

def extract_features(image_path: Path) -> dict | None:
    """Run all four forensic modules and return a flat feature dict."""
    try:
        path = str(image_path)

        # ELA
        _, _, ela = compute_ela(path, resave_quality=75, amplify=15.0)

        # EXIF
        exif = extract_exif(path)
        exif_flags = exif.get("flags", [])
        has_make_model = int(
            bool(exif.get("raw", {}).get("Make")) and
            bool(exif.get("raw", {}).get("Model"))
        )
        timestamp_mismatch = int(
            any("Timestamp mismatch" in f for f in exif_flags)
        )

        # Noise
        _, _, noise = noise_consistency_map(path, tile_size=32)

        # FFT
        _, _, fft = fft_spectrum_analysis(path)

        return {
            # ELA
            "ela_mean":               ela["mean_ela"],
            "ela_max":                ela["max_ela"],
            "ela_std":                ela["std_ela"],
            "ela_regional_variance":  ela["regional_variance"],
            "ela_suspicious_pct":     ela["suspicious_pixel_pct"],
            # EXIF
            "exif_risk_score":        EXIF_RISK_MAP.get(exif["risk_level"], 0.5),
            "exif_flag_count":        len(exif_flags),
            "exif_has_make_model":    has_make_model,
            "exif_timestamp_mismatch": timestamp_mismatch,
            # Noise
            "noise_median_sigma":     noise["median_noise_sigma"],
            "noise_std_sigma":        noise["std_noise_sigma"],
            "noise_suspicious_pct":   noise["suspicious_pct"],
            # FFT
            "fft_high_freq_ratio":   fft["high_freq_energy_ratio"],
            "fft_peak_freq_bin":     fft["peak_freq_bin"],
        }
    except Exception as e:
        print(f"  ⚠  Skipping {image_path.name}: {e}")
        return None


records = []

print("Extracting features from AUTHENTIC images...")
for p in sorted(ORIGINAL_DIR.iterdir()):
    if p.suffix.lower() in SUPPORTED:
        feat = extract_features(p)
        if feat:
            feat["label"] = 0   # 0 = AUTHENTIC
            feat["file"]  = p.name
            records.append(feat)
            print(f"  ✓ {p.name}")

print("\nExtracting features from TAMPERED images...")
for p in sorted(TAMPERED_DIR.iterdir()):
    if p.suffix.lower() in SUPPORTED:
        feat = extract_features(p)
        if feat:
            feat["label"] = 1   # 1 = MANIPULATED
            feat["file"]  = p.name
            records.append(feat)
            print(f"  ✗ {p.name}")

df = pd.DataFrame(records)
print(f"\nDataset: {len(df)} images | {df['label'].value_counts().to_dict()} (0=authentic, 1=tampered)")
df.head()

## 2. Data Exploration — Feature Distributions

In [ ]:
# ── 2. Feature Distribution Plots ───────────────────────────────────────────
FEATURES = [
    "ela_mean", "ela_std", "ela_regional_variance", "ela_suspicious_pct",
    "exif_risk_score", "exif_flag_count",
    "noise_median_sigma", "noise_std_sigma", "noise_suspicious_pct",
    "fft_high_freq_ratio",
]

palette = {0: "#4CAF50", 1: "#F44336"}   # green = authentic, red = tampered
label_names = {0: "Authentic", 1: "Tampered"}
df["label_name"] = df["label"].map(label_names)

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    for lbl, grp in df.groupby("label_name"):
        axes[i].hist(
            grp[feat], bins=15, alpha=0.6,
            label=lbl, color=palette[0] if lbl == "Authentic" else palette[1]
        )
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=8)

plt.suptitle("Feature Distributions: Authentic vs Tampered", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "feature_distributions.png", dpi=150)
plt.show()
print("Saved → reports/feature_distributions.png")

In [ ]:
# ── 2b. Correlation Heatmap ──────────────────────────────────────────────────
numeric_df = df[FEATURES + ["label"]].copy()
corr = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, linewidths=0.4, annot_kws={"size": 7}
)
plt.title("Feature Correlation Matrix (incl. label)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "correlation_heatmap.png", dpi=150)
plt.show()
print("Saved → reports/correlation_heatmap.png")

## 3. Model Training

We train three classifiers and compare them using 5-fold stratified cross-validation:
- **Logistic Regression** (baseline)
- **Random Forest** (ensemble, handles non-linearity)
- **Gradient Boosting** (sequential ensemble, often best on tabular data)

> **Note:** With small datasets (< 100 images), cross-validation scores may vary. The more labeled images you add to `samples/`, the better the model will generalize.

In [ ]:
# ── 3. Model Training ────────────────────────────────────────────────────────
X = df[FEATURES].values
y = df["label"].values

if len(df) < 6:
    print("⚠  Too few samples for a reliable train/test split.")
    print("   Add more images to samples/original/ and samples/tampered/ then re-run.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    candidates = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=500, random_state=42)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, max_depth=8,
            class_weight="balanced", random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=150, max_depth=4,
            learning_rate=0.05, random_state=42
        ),
    }

    results = {}
    for name, model in candidates.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
        results[name] = scores
        print(f"  {name:25s}  CV F1: {scores.mean():.3f} ± {scores.std():.3f}")

    best_name = max(results, key=lambda n: results[n].mean())
    print(f"\n✅ Best model: {best_name}")

## 4. Evaluation — Test Set

In [ ]:
# ── 4. Evaluation on Hold-out Test Set ──────────────────────────────────────
best_model = candidates[best_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"\n── {best_name} — Test Set Results ──")
print(classification_report(y_test, y_pred, target_names=["Authentic", "Tampered"]))

# ── Confusion Matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Authentic", "Tampered"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

# ── ROC Curve ────────────────────────────────────────────────────────────────
if hasattr(best_model, "predict_proba"):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color="#E91E63", lw=2, label=f"AUC = {roc_auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve")
    axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / "reports" / "model_evaluation.png", dpi=150)
plt.show()
print("Saved → reports/model_evaluation.png")

In [ ]:
# ── 4b. Feature Importance (Random Forest / GB only) ────────────────────────
clf = best_model[-1] if hasattr(best_model, "__getitem__") else best_model

if hasattr(clf, "feature_importances_"):
    importances = pd.Series(clf.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)

    plt.figure(figsize=(9, 6))
    importances.plot(kind="barh", color="#7C4DFF")
    plt.xlabel("Importance")
    plt.title(f"Feature Importances — {best_name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(ROOT / "reports" / "feature_importance.png", dpi=150)
    plt.show()
    print("Saved → reports/feature_importance.png")
    print("\nTop features:")
    print(importances.sort_values(ascending=False).head(5))

## 5. Heuristic Weight Calibration (Optional)

Use this cell to visualise the **separation score** for each feature — this directly tells you which signals deserve more weight in `analyzer.py`'s `WEIGHTS` dict.

In [ ]:
# ── 5. Heuristic Signal Separation Analysis ──────────────────────────────────
auth  = df[df["label"] == 0]
tamp  = df[df["label"] == 1]

sep_rows = []
for feat in FEATURES:
    mean_auth  = auth[feat].mean()
    mean_tamp  = tamp[feat].mean()
    separation = mean_tamp - mean_auth
    sep_rows.append({
        "feature":     feat,
        "mean_auth":   round(mean_auth,  4),
        "mean_tamp":   round(mean_tamp,  4),
        "separation":  round(separation, 4),
    })

sep_df = pd.DataFrame(sep_rows).sort_values("separation", ascending=False)

plt.figure(figsize=(10, 5))
colors = ["#4CAF50" if v > 0 else "#F44336" for v in sep_df["separation"]]
plt.barh(sep_df["feature"], sep_df["separation"], color=colors)
plt.axvline(0, color="white", lw=1)
plt.xlabel("Mean(tampered) − Mean(authentic)")
plt.title("Signal Separation: Which features best detect tampering?", fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "signal_separation.png", dpi=150)
plt.show()
print(sep_df.to_string(index=False))

## 6. Export Model

Save the trained model to `models/forensics_model.joblib`.

To use it in `analyzer.py`, replace the `compute_verdict` weighted-sum logic with:
```python
import joblib
model = joblib.load("models/forensics_model.joblib")
score = model.predict_proba([[...features...]])[0][1]
```

In [ ]:
# ── 6. Save Model ────────────────────────────────────────────────────────────
MODEL_PATH = MODEL_DIR / "forensics_model.joblib"

# Re-fit on the full dataset before saving
best_model.fit(X, y)
joblib.dump(best_model, MODEL_PATH)

print(f"✅ Model saved → {MODEL_PATH}")
print(f"   Type        : {best_name}")
print(f"   Features    : {FEATURES}")
print(f"   Trained on  : {len(df)} images")

# Save feature list alongside the model for inference consistency
meta = {"model": best_name, "features": FEATURES, "n_samples": len(df)}
(MODEL_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
print("   Metadata    : models/model_meta.json")